In [4]:
import pandas as pd

# 1. Load Geolocation Map File
df = pd.read_csv("../data/processed/fraud_with_country.csv")

# 2. Datetime Parsing
df['signup_time'] = pd.to_datetime(df['signup_time'])
df['purchase_time'] = pd.to_datetime(df['purchase_time'])

# 3. Derive Temporal Spacing Features
df['time_since_signup'] = (df['purchase_time'] - df['signup_time']).dt.total_seconds()
df['hour_of_day'] = df['purchase_time'].dt.hour
df['day_of_week'] = df['purchase_time'].dt.dayofweek

# 4. Compute Transaction Velocity (Fixes Duplicate Timestamps)
# Ensure the dataset is perfectly sorted chronologically first
df = df.sort_values(by='purchase_time').reset_index(drop=True)

# Calculate rolling count per device_id over a 30-minute time window
velocity_series = (
    df.set_index('purchase_time')
    .groupby('device_id')
    .rolling(window='30min')
    .count()['user_id']
)

# Align the calculated values back to the original rows safely using multi-index mapping
df = df.set_index(['device_id', 'purchase_time'])
df['device_velocity_30m'] = velocity_series
df = df.reset_index()

# 5. Save Final Engineered E-Commerce Dataset
df.to_csv("../data/processed/fraud_final.csv", index=False)
print("Task 1 part 2 complete: E-commerce features engineered with zero errors!")



Task 1 part 2 complete: E-commerce features engineered with zero errors!
